# Deliverable 3: Our First Model, Classification with KNN

**MOD10 Introduction to Machine Learning**, November 2026

- **Group:** CUPILLARD Eliot, DENOUAL Romain, ROY-NOUGUIER Erwann
- **Dataset:** Arbres publics sur le territoire de la Ville (consolidated file), City of Montreal, licence CC BY 4.0, same unchanged copy as Deliverable 2 (https://doi.org/10.5281/zenodo.23540888)
- **Category (`y`):** trunk size class (young, medium, mature) built from `DHP`, the trunk diameter at 1.4 m
- **Question:** Can KNN tell the size class of a public tree from its species, location and age?

## 0. Setup and data from Deliverable 2

This notebook must run on its own, so the cells below repeat the Deliverable 2 pipeline in short form: download, cleaning, then encoding of the features. The reasons for each choice are in the Deliverable 2 notebook.

The encoded features are in `X`. The real target, the trunk diameter `DHP`, stays in `df_clean` and is not in `X`.

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

SEED = 42

PROCESSED_DATA_PATH = Path("../data/processed/trees_encoded.csv")
df_clean = pd.read_csv(PROCESSED_DATA_PATH, index_col=0)


In [ ]:
print(f"Data shape: {df_clean.shape}")

## 1. Our category

Our real target `DHP` is a number, and KNN needs a category. So we build one from `DHP`: the size class of the trunk.

| Class | DHP |
|---|---|
| `young` | up to 10 |
| `medium` | above 10, up to 30 |
| `mature` | above 30 |

- **Cut points:** round values, close to the thirds of the data (7 and 32). They give three groups of similar size (39%, 27%, 35%). The choice is ours: we did not use an official scale.
- **Edges:** a tree with a DHP of exactly 10 is `young`, exactly 30 is `medium` (6,107 and 4,777 trees).
- **Limit 1:** `DHP` is a trunk diameter, not an age. A fast-growing species can be `mature` while still young, so `mature` means a large trunk.
- **Limit 2:** 70% of the `young` trees have a DHP of exactly 5, probably a default value (see Deliverable 2). Part of this class is "size not really measured".
- `DHP` itself will not be in the features, because the category comes from it.

In [9]:
y = pd.cut(df_clean["DHP"], bins=[0, 10, 30, 250], labels=["young", "medium", "mature"])
df_clean["DHP"].groupby(y, observed=True).agg(["count", "min", "median", "max"])

,count,min,median,max
DHP,,,,
young,129358,1.0,5.0,10.0
medium,89385,10.1,21.0,30.0
mature,116207,30.1,47.0,246.0


## 2. Features and label

`y` is the size class of section 1, and `X` is the encoded table of section 0. Two points to check:

- `DHP` is not in `X`: the category comes from it, so keeping it would be data leakage. The identifiers and the columns that repeat other information (a code, a translation, the same position in another system) are not in `X` either.
- Everything in `X` is a number, with no missing value.

**Class balance:** the classes are not equal, but none is rare. A model that always answers `young` would be right 38.6% of the time. This is the baseline our KNN has to beat.

In [10]:
print("X:", X.shape, "| y:", y.shape)
print("Columns that are not numbers:", list(X.select_dtypes(exclude="number").columns))
print("Missing values:", X.isna().sum().sum())
pd.DataFrame({"trees": y.value_counts(), "share": y.value_counts(normalize=True).round(3)}).sort_index()

X: (334950, 61) | y: (334950,)
Columns that are not numbers: []
Missing values: 0


,trees,share
DHP,,
young,129358,0.386
medium,89385,0.267
mature,116207,0.347


## 3. Split and scale

*TODO: say that we reuse the Deliverable 2 split.*

- Same `test_size=0.2` and `random_state=SEED` as Deliverable 2, so the rows in train and test are the same.
- KNN measures distances, so the numeric columns (`Latitude`, `Longitude`, `age`) are standardized. The scaler is fitted on the training set only.

In [ ]:
# TODO: train_test_split, then StandardScaler on ["Latitude", "Longitude", "age"] (fit on train only)

## 4. Train KNN

*TODO: explain the choice of the values of k.*

- Starting point from the course: k close to the square root of the number of training rows (about 517 here).
- Compare small and large values of k. A small k follows the noise (overfitting), a large k is too smooth (underfitting).
- KNN compares each test tree with all training trees, so each k takes about a minute on the whole test set.
- How we choose k: *TODO (to be decided: test set as in the course, or a validation set taken from the training set).*

In [ ]:
# TODO: loop over several values of k, keep the accuracy of each

*TODO: table of k and accuracy, and the value of k we keep, with the reason.*

## 5. Evaluation

*TODO: final model with the chosen k. Report the accuracy and the confusion matrix on the test set.*

- Rows are the real classes, columns are the predictions.
- Compare the accuracy with the baseline: always answering `young` gives 38.6%.

In [ ]:
# TODO: predictions of the final model, accuracy_score and confusion_matrix(labels=["young", "medium", "mature"])

## 6. Interpretation

*TODO: what the mistakes mean for our problem, and whether accuracy is enough.*

- Which classes does the model mix up? The classes are ordered: `young` to `medium` is a smaller mistake than `young` to `mature`.
- Is one mistake worse than another for a city that uses the inventory (planning of pruning, replacement, canopy)?
- Are the classes balanced enough for accuracy alone to be a fair measure?

## 7. Model fit

*TODO: one line for the final presentation: does KNN suit this project, or would another model be better? (See the Choosing the Right Model guide.)*

Points to check: size of the data (335,000 trees, 61 columns), most columns are 0/1, and the real target is a number.